### Estimating Polynya Area and HSSW Production Rate from Surface Heat Fluxes 

In [ ]:
import xarray as xr 
import numpy as np 
import matplotlib.pyplot as plt 
import glob

### Load in SIC data 

In [ ]:
files = glob.glob(
    r"C:\Users\mblan\Documents\TNB_Research\data\SIC_Ross\**\*.nc",
    recursive=True
)

In [ ]:
files

In [ ]:
import glob
import xarray as xr

files = glob.glob(
    r"C:\Users\mblan\Documents\TNB_Research\data\SIC_Ross\**\*.nc",
    recursive=True
)

sic = xr.open_mfdataset(
    files,
    combine="by_coords"
)

print(sic)

In [ ]:
one = xr.open_dataset(files[1])


In [ ]:
print(one)

In [ ]:
import xarray as xr
import pandas as pd
import glob
import re
import numpy as np

In [ ]:
def get_time_from_filename(fname):
    match = re.search(r'(\d{8})', fname)
    if match is None:
        raise ValueError(f"No date found in filename: {fname}")
    return pd.to_datetime(match.group(1), format='%Y%m%d')

In [ ]:
files = glob.glob(
    r"C:\Users\mblan\Documents\TNB_Research\data\SIC_Ross\**\*.nc",
    recursive=True
)

datasets = []

for f in files:
    ds = xr.open_dataset(f)

    # rename variable if needed
    ds = ds.rename({"z": "sic"})  # sea ice concentration

    # add time dimension
    time = get_time_from_filename(f)
    ds = ds.expand_dims({"time": [time]})

    datasets.append(ds)

In [ ]:
len(datasets)

In [ ]:
datasets[9]

In [ ]:
ds_all = xr.concat(datasets, dim="time")

In [ ]:
ds_all

In [ ]:
# rough Ross Sea bounds (you may tweak)
x_min, x_max = -2.2e6, 1.5e6
y_min, y_max = -2.2e6, -0.2e6

In [ ]:
ds_ross = ds_all.where(
    (ds_all.x >= x_min) &
    (ds_all.x <= x_max) &
    (ds_all.y >= y_min) &
    (ds_all.y <= y_max),
    drop=True
)

In [ ]:
ds_ross

In [ ]:
ds_ross.sic.isel(time=500).plot(
    cmap=cmocean.cm.ice,
    vmin=0,
    vmax=100,
    figsize=(8,8)
)

plt.title(str(ds_ross.time.isel(time=0).values))
plt.show()

In [ ]:
x_min = 0.2e6
x_max = 0.9e6

y_min = -2.5e6
y_max = -1.6e6

In [ ]:
ds_tnb = ds_ross.where(
    (ds_ross.x >= x_min) &
    (ds_ross.x <= x_max) &
    (ds_ross.y >= y_min) &
    (ds_ross.y <= y_max),
    drop=True
)

In [ ]:
ds_tnb.sic.isel(time=500).plot(
    cmap=cmocean.cm.ice,
    vmin=0,
    vmax=100,
    figsize=(8,8)
)

plt.title(str(ds_tnb.time.isel(time=0).values))
plt.show()

In [ ]:
# # load in SIC data 
# # open TNB dataset - MAC
# tnb = xr.open_dataset('/Users/mackenzie/Documents/Research/PhD/TNB_Research/data/SIC_TNB_2016_2019.nc')

# open TNB dataset - DELL 
tnb = xr.open_dataset('/Users/mblan/Documents/TNB_Research/data/SIC_Ross/SIC_TNB_2016_2019.nc')

In [ ]:
tnb

In [ ]:
fig = plt.figure()
plt.pcolormesh(tnb.lon,tnb.lat,tnb.sic.isel(time=0))
plt.colorbar()
plt.xlim(163,165.5)
plt.ylim(-75.4,-74.5)

In [ ]:
subset = tnb.where(
    (tnb.lon >= 163) &
    (tnb.lon <= 165) &
    (tnb.lat >= -75.2) &
    (tnb.lat <= -74.7),
    drop=True
)

In [ ]:
subset

In [ ]:
fig = plt.figure()
plt.pcolormesh(subset.lon,subset.lat,subset.sic.isel(time=0))
plt.colorbar()
# plt.xlim(163,165.5)
# plt.ylim(-75.4,-74.5)

In [ ]:
# estimate polynya area 
threshold = 30 #percent 

In [ ]:
day = subset.isel(time=0)

In [ ]:
sic_day = day.where(day.sic > 30, drop=True).sic

In [ ]:
np.nancumsum(sic_day)

In [ ]:
# Counts all valid numbers (Returns: 4)
pixel_count = np.sum(~np.isnan(sic_day))

In [ ]:
polynya_area = pixel_count * 3.125**2

In [ ]:
area = []
for day in range(len(subset.time)):
    d = subset.isel(time=day)
    sic_day = d.where(d.sic > 30, drop=True).sic
    pixel_count = np.sum(~np.isnan(sic_day))
    polynya_area = pixel_count * 3.125**2
    area.append(polynya_area)

In [ ]:
len(area)

In [ ]:
polynya_area = xr.concat(area,dim='time')

In [ ]:
fig = plt.figure(figsize=(12,4))
plt.plot(polynya_area.time,polynya_area)
plt.ylabel('Polynya Area [km^2]')


In [ ]:
area_2018 = polynya_area.sel(time=slice('2018-01','2018-12'))

In [ ]:
area_2018

In [ ]:
fig = plt.figure(figsize=(12,4))
plt.plot(area_2018.time,area_2018)
plt.ylabel('Polynya Area [km^2]')

In [ ]:
polynya_area

In [ ]:
area_2018.to_netcdf("polynya_area_2018.nc")